In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from broinsight.tools.register import tool, ToolBox, TypeValidator

In [3]:
from datetime import datetime
@tool()
def add(a:float, b:float)->float:
    """Add two numbers together
    Keywords: math, calculation, arithmetic, sum
    Args:
        a (float) : a number
        b (float) : a number
    Returns:
        float : a scalar number
    """
    return a + b

@tool()
def add_calendar(event_name:str, datetime:datetime)->str:
    """Add event into calendar
    Keywords: calendar, schedule, appointment, meeting, event
    Category: scheduling
    Args:
        event_name (str) : name of the event
        datetime (datetime) : a datetime of the event
    Returns:
        str : successfully added message
    """
    return f"{event_name} at {datetime}"

@tool()
def check_stock_index(index:str)->str:
    """Check a stock by index from Yahoo API
    Keywords: stock, investment, finance
    Args:
        index (str) : a stock index
    Returns:
        str : a response from Yahoo API
    """
    return index

In [4]:
ToolBox.list_tools()

{'add': {'function': <function __main__.add(a: float, b: float) -> float>,
  'name': 'add',
  'description': 'Add two numbers together',
  'category': 'general',
  'keywords': ['math', 'calculation', 'arithmetic', 'sum'],
  'parameters': {'a': 'float - a number', 'b': 'float - a number'},
  'returns': 'float - a scalar number',
  'signature': <Signature (a: float, b: float) -> float>},
 'add_calendar': {'function': <function __main__.add_calendar(event_name: str, datetime: datetime.datetime) -> str>,
  'name': 'add_calendar',
  'description': 'Add event into calendar',
  'category': 'scheduling',
  'keywords': ['calendar', 'schedule', 'appointment', 'meeting', 'event'],
  'parameters': {'event_name': 'str - name of the event',
   'datetime': 'datetime - a datetime of the event'},
  'returns': 'str - successfully added message',
  'signature': <Signature (event_name: str, datetime: datetime.datetime) -> str>},
 'check_stock_index': {'function': <function __main__.check_stock_index(index

In [5]:
from broinsight.core.llm import LocalOpenAI, UserMessage, AIMessage, ModelResponse
from broinsight.utils.parse_string import parse_json, parse_blockcode

In [6]:
tool_call = ToolBox.get_tool_info("add_calendar")
name = tool_call['name']
function = tool_call['function']
description = tool_call['description']
parameters = tool_call["parameters"]

tool_prompt = """\
Tool: {name}
Description: {description}
Parameters: {parameters}
""".strip().format(name=name, description=description, parameters=parameters)
print(tool_prompt)

Tool: add_calendar
Description: Add event into calendar
Parameters: {'event_name': 'str - name of the event', 'datetime': 'datetime - a datetime of the event'}


In [7]:
system_prompt = """\
- Extract input parameters from USER_INPUT based on provided TOOLS
- Return in JSON codeblock that matches the TOOLS
- Return only the JSON codeblock
```json
{"parameters": {"key": value, "key", value}}
```
""".strip()

model = LocalOpenAI()
# user_input = "what is twenty two plus one?"
user_input = "I wanna see grandma on 10 October 2025 at noon."
# user_input = "I wanna know how Apple stock goes?"
content = f"TOOLS:\n\n{tool_prompt}\n\nUSER_INPUT:\n\n{user_input}\n\n"
response = model.run(system_prompt, messages=[UserMessage(content=content)])

In [8]:
import json
params = json.loads(parse_json(response.content))
print(params)

print(function(**params['parameters']))

{'parameters': {'event_name': 'see grandma', 'datetime': '2025-10-10T12:00:00'}}
see grandma at 2025-10-10T12:00:00


In [9]:
# bound_args = sig.bind(**params['parameters'])
signature = tool_call["signature"]
bound_args = signature.bind(**params["parameters"])
bound_args.apply_defaults()

In [10]:
validator = TypeValidator()
valid_parameters = validator.validate_types(bound_args, signature)
valid_parameters

{'event_name': 'see grandma',
 'datetime': datetime.datetime(2025, 10, 10, 12, 0)}

In [11]:
import torch
torch.cuda.is_available()

True

In [ ]:
from sentence_transformers import CrossEncoder

# 1. Load a pretrained CrossEncoder model
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

d:\broinsight\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
ToolBox.list_tools()

{'add': {'function': <function __main__.add(a: float, b: float) -> float>,
  'name': 'add',
  'description': 'Add two numbers together',
  'category': 'general',
  'keywords': ['math', 'calculation', 'arithmetic', 'sum'],
  'parameters': {'a': 'float - a number', 'b': 'float - a number'},
  'returns': 'float - a scalar number',
  'signature': <Signature (a: float, b: float) -> float>},
 'add_calendar': {'function': <function __main__.add_calendar(event_name: str, datetime: datetime.datetime) -> str>,
  'name': 'add_calendar',
  'description': 'Add event into calendar',
  'category': 'scheduling',
  'keywords': ['calendar', 'schedule', 'appointment', 'meeting', 'event'],
  'parameters': {'event_name': 'str - name of the event',
   'datetime': 'datetime - a datetime of the event'},
  'returns': 'str - successfully added message',
  'signature': <Signature (event_name: str, datetime: datetime.datetime) -> str>},
 'check_stock_index': {'function': <function __main__.check_stock_index(index

In [14]:
tools = ToolBox.list_tool_metadata()
tools

["add Add two numbers together general ['math', 'calculation', 'arithmetic', 'sum']",
 "add_calendar Add event into calendar scheduling ['calendar', 'schedule', 'appointment', 'meeting', 'event']",
 "check_stock_index Check a stock by index from Yahoo API general ['stock', 'investment', 'finance']"]

In [ ]:
# user_input = "I wanna know how Apple stock goes?"
# user_input = "What is one plus one?"
user_input = "I wanna see grandma on 10 October 2025 at noon."
scores = reranker.predict([(user_input, t) for t in tools])
scores

array([-11.261845, -11.341116, -11.356722], dtype=float32)

In [ ]:
ranks = reranker.rank(user_input, tools, return_documents=True)
ranks[:5]

[{'corpus_id': 0,
  'score': np.float32(-11.261845),
  'text': "add Add two numbers together general ['math', 'calculation', 'arithmetic', 'sum']"},
 {'corpus_id': 1,
  'score': np.float32(-11.341116),
  'text': "add_calendar Add event into calendar scheduling ['calendar', 'schedule', 'appointment', 'meeting', 'event']"},
 {'corpus_id': 2,
  'score': np.float32(-11.356722),
  'text': "check_stock_index Check a stock by index from Yahoo API general ['stock', 'investment', 'finance']"}]

In [17]:
candidated_tools = [r['text'].split(" ")[0] for r in ranks[:5]]
candidated_tools

['add', 'add_calendar', 'check_stock_index']

In [18]:
ToolBox.get_tools(candidated_tools)

[{'function': <function __main__.add(a: float, b: float) -> float>,
  'name': 'add',
  'description': 'Add two numbers together',
  'category': 'general',
  'keywords': ['math', 'calculation', 'arithmetic', 'sum'],
  'parameters': {'a': 'float - a number', 'b': 'float - a number'},
  'returns': 'float - a scalar number',
  'signature': <Signature (a: float, b: float) -> float>},
 {'function': <function __main__.add_calendar(event_name: str, datetime: datetime.datetime) -> str>,
  'name': 'add_calendar',
  'description': 'Add event into calendar',
  'category': 'scheduling',
  'keywords': ['calendar', 'schedule', 'appointment', 'meeting', 'event'],
  'parameters': {'event_name': 'str - name of the event',
   'datetime': 'datetime - a datetime of the event'},
  'returns': 'str - successfully added message',
  'signature': <Signature (event_name: str, datetime: datetime.datetime) -> str>},
 {'function': <function __main__.check_stock_index(index: str) -> str>,
  'name': 'check_stock_index

In [24]:
ToolBox.prompt(candidated_tools)

["- add: Add two numbers together (parameters: ['a', 'b'])",
 "- add_calendar: Add event into calendar (parameters: ['event_name', 'datetime'])",
 "- check_stock_index: Check a stock by index from Yahoo API (parameters: ['index'])"]

In [30]:
from broinsight.core.llm import LocalOpenAI, ModelResponse, UserMessage, AIMessage
from broinsight.prompt_hub import PromptHub
llm = LocalOpenAI()

# user_input = "I wanna know how Apple stock goes?"
# user_input = "I wanna see grandma on 10 October 2025 at noon."
# user_input = "What is 1 + 1?"
# user_input = "Call Mary for me."
# user_input = "Tell me a joke."
user_input = "How's the weather in New York?"
content = f"AVAILABLE_TOOLS:\n\n{ToolBox.prompt(candidated_tools)}\n\nUSER_INPUT:\n\n{user_input}\n\n"
response = llm.run(PromptHub().tool_selection, messages=[UserMessage(content=content)])
print(response.content)

```json
{"selected_tool": null}
```




In [43]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load once, reuse many times
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_similarity(user_input: str, tool_texts: list) -> list:
    # Encode all at once (faster than individual calls)
    all_texts = [user_input] + tool_texts
    embeddings = embedder.encode(all_texts)
    
    user_emb = embeddings[0]
    tool_embs = embeddings[1:]
    
    # Cosine similarity
    similarities = np.dot(tool_embs, user_emb) / (
        np.linalg.norm(tool_embs, axis=1) * np.linalg.norm(user_emb)
    )
    
    return similarities


In [48]:
# user_input = "I wanna know how Apple stock goes?"
# user_input = "I wanna see grandma on 10 October 2025 at noon."
user_input = "What is 1 + 1?"
# user_input = "Call Mary for me."
# user_input = "Tell me a joke."
# user_input = "How's the weather in New York?"

scores = calculate_similarity(user_input, tools)
scores

array([ 0.36557853,  0.08871193, -0.00741214], dtype=float32)

In [49]:
any(scores > 0.1)

True